# FHVHV EDA: automatic reports with Sweetviz

A first look at each column (distributions, nulls, associations) generated automatically as HTML reports in the `reports/` folder.

**Why Sweetviz:** `ydata-profiling` does not support Python 3.14 yet, and `skimpy` breaks with pandas 3. Sweetviz 2.3 works with both.

**Limitations:**
- It runs on a **random sample**. Rare events (3 duplicates, 313 negative `driver_pay`) may not show up. Those are covered by `02_eda_quality.ipynb`.
- It looks at columns one by one or in pairs. It does not know business rules such as "pickup must come after request".

## 1. Setup

In [ ]:
import logging
from pathlib import Path

import pandas as pd
import sweetviz as sv

# Matplotlib prints hundreds of harmless font warnings during the reports
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

data_dir = Path("datasets")
reports_dir = Path("reports")
reports_dir.mkdir(exist_ok=True)

SAMPLE_SIZE = 300_000  # bigger = more accurate but slower
MONTH = "2025-01"
COMPARE_MONTH = "2024-01"

## 2. Load a sample and derive readable features

Sweetviz does not handle datetime columns well, so they are replaced by features that are easier to read: pickup hour, weekday, waiting time and trip duration in minutes.

In [ ]:
COMPANIES = {"HV0002": "Juno", "HV0003": "Uber", "HV0004": "Via", "HV0005": "Lyft"}
DATETIME_COLS = ["request_datetime", "on_scene_datetime", "pickup_datetime", "dropoff_datetime"]

def load_sample(month: str, size: int = SAMPLE_SIZE) -> pd.DataFrame:
    full = pd.read_parquet(data_dir / f"fhvhv_tripdata_{month}.parquet")
    df = full.sample(size, random_state=42)
    del full  # free the full month right away

    df["company"] = df["hvfhs_license_num"].map(COMPANIES)
    df["pickup_hour"] = df["pickup_datetime"].dt.hour
    df["pickup_weekday"] = df["pickup_datetime"].dt.day_name()
    df["wait_minutes"] = (df["pickup_datetime"] - df["request_datetime"]).dt.total_seconds() / 60
    df["trip_minutes"] = df["trip_time"] / 60

    return df.drop(columns=DATETIME_COLS + ["hvfhs_license_num", "trip_time"]).reset_index(drop=True)

sample = load_sample(MONTH)
print(f"{MONTH}: sample of {len(sample):,} rows x {sample.shape[1]} columns")
sample.head()

In [ ]:
# Treat as categories:
# - zone IDs and pickup hour: they are numbers, but labels, not quantities
# - text columns: without this, Sweetviz sees a Y/N flag as boolean on one side and as
#   text on the other (e.g. Lyft only has "N" in shared_request_flag) and the comparisons fail
TEXT_COLS = sample.select_dtypes(include=["object", "string", "category"]).columns.tolist()
feature_config = sv.FeatureConfig(force_cat=TEXT_COLS + ["PULocationID", "DOLocationID", "pickup_hour"])
print("Forced as categorical:", feature_config.force_cat)

## 3. Report for one month

Open the HTML file in the browser. Click a column on the left to see its distribution, top values and associations with the other columns.

In [ ]:
report = sv.analyze(sample, feat_cfg=feature_config)
path = reports_dir / f"sweetviz_{MONTH}.html"
report.show_html(str(path), open_browser=False)
print("Open:", path.resolve())

## 4. Compare Uber vs Lyft (same month)

Splits the sample by company. Look at `originating_base_num`: the Lyft nulls found in the manual EDA should stand out here. Also compare `wait_minutes`, fares and `driver_pay` between the two companies.

In [ ]:
report = sv.compare_intra(sample, sample["company"] == "Uber", ["Uber", "Lyft"], feat_cfg=feature_config)
path = reports_dir / f"sweetviz_{MONTH}_uber_vs_lyft.html"
report.show_html(str(path), open_browser=False)
print("Open:", path.resolve())

## 5. Compare two months

Shows how the distributions changed between the two months. `cbd_congestion_fee` only exists in 2025, so it appears in one side only.

In [ ]:
other = load_sample(COMPARE_MONTH)

report = sv.compare([sample, MONTH], [other, COMPARE_MONTH], feat_cfg=feature_config)
path = reports_dir / f"sweetviz_{MONTH}_vs_{COMPARE_MONTH}.html"
report.show_html(str(path), open_browser=False)
print("Open:", path.resolve())

## 6. What to look for in the reports

- **Missing values** per column, and whether they differ between Uber and Lyft.
- **Distributions** with long tails (`trip_miles`, `base_passenger_fare`, `trip_minutes`) and negative minimums.
- **`wait_minutes` negative**: the "pickup before request" case from the manual EDA.
- **Associations** (the grid at the top): which columns move together, e.g. fare × miles × minutes.
- **Month comparison**: changes in fare, tolls or congestion fees between 2024 and 2025.

Anything new you find here should become a check in `02_eda_quality.ipynb`, where it is measured on the full month.